# Transform Constructors  Data

1. Read bronze `constructors` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`)
1. Rename columns to make them more meaningful (`name` → `constructor_name`)
1. Remove duplicate records
1. Transform values of column `nationality` to Title Case
1. Write the transformed data to silver `constructors` table


#### Entity Relationship Diagram - Formula1 Schema

![Formula1 Raw Data.png](../../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")



In [0]:
%run "../00-common/01.environment-config"


In [0]:
val bronze_table = catalog_name + "." + bronze_schema + "." + "constructors "
val silver_table = catalog_name + "." + silver_schema + "." + "constructors"

#### Step 1 - Read bronze `constructors` table

In [0]:
val constructors_df= spark.table(bronze_table)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
val constructors_selected_df = constructors_df.select(
"constructorId",
"name",
"nationality",
"ingestion_timestamp",
"source_file"
)

#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (constructorId → constructor_id)
- Rename columns to make them more meaningful (name → constructor_name)


In [0]:
val constructors_renamed_df = constructors_selected_df
.withColumnRenamed("constructorId","constructor_id")
.withColumnRenamed("name", "constructor_name")

#### Step 5 - Remove duplicate records

In [0]:
val constructors_distinct_df = constructors_renamed_df.dropDuplicates("constructor_id")

In [0]:
display(constructors_distinct_df)

#### Step 7 - Transform values of columns `nationality`  to Title Case


In [0]:
import org.apache.spark.sql.functions.{initcap,col}
val constructors_final_df= constructors_distinct_df
.withColumn("nationality",initcap(col("nationality")))

In [0]:
display(constructors_final_df)

#### Step 8 - Write the transformed data to silver `constructors` table

In [0]:
constructors_final_df.write.format("delta").mode("overwrite")
        .saveAsTable(silver_table)


In [0]:
display(spark.table(silver_table))